# Supervised vs Unsupervised Learning

**Topic:** Learning Paradigms

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Distinguish** between supervised and unsupervised learning by looking at whether labels are present
- **Explain** why the same dataset looks different depending on which paradigm you apply
- **Identify** which learning paradigm is appropriate for a given real-world problem

> **Tip:** Start with **Classification**, then switch to **No labels**. Notice how the same data looks like two clean groups in one view and an unlabeled cloud in the other. The data did not change — only the available information did.

---
## How we got here

In **01_what_is_machine_learning.ipynb** we established that machine learning learns rules from data rather than following rules written by humans. Now we ask: what kind of data does it learn from?

The answer splits machine learning into two major families. In one, a teacher provides correct answers during training. In the other, the algorithm must find structure on its own, with no guidance at all. Understanding which world you are in is the first decision in any ML project.

---
## Why this matters for data science

The choice between supervised and unsupervised learning is not a technical preference — it is determined by your data. If you have labeled examples (past fraud cases, diagnosed patients, approved/rejected loans), you use supervised learning. If you only have raw observations with no labels, you must find patterns without guidance.

Getting this wrong wastes months. Trying to cluster customer behavior when you actually have purchase labels — or building a classifier when no one has labeled your data yet — are common and expensive mistakes.

---
## Try it yourself

In [2]:
np.random.seed(42)
X_cls, y_cls = make_classification(n_samples=200, n_features=2,
    n_informative=2, n_redundant=0, random_state=42)
X_moons, _ = make_moons(n_samples=200, noise=0.15, random_state=42)
X_reg = np.linspace(0, 10, 80).reshape(-1, 1)
y_reg = 2*X_reg.ravel() + np.random.normal(0, 2, 80)

out = Output()

paradigm_dd = Dropdown(
    options=[("Supervised — Classification", "clf"),
             ("Supervised — Regression",     "reg"),
             ("Unsupervised — Clustering",   "clu"),
             ("Unsupervised — No labels",    "raw")],
    value="clf",
    description="Paradigm:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="400px"))

def render(mode):
    if mode == "clf":
        colors = [PALETTE["primary"] if yi==1 else PALETTE["secondary"] for yi in y_cls]
        traces = [go.Scatter(x=X_cls[:,0], y=X_cls[:,1], mode="markers",
            marker=dict(color=colors, size=8, opacity=0.7), showlegend=False)]
        layout = base_layout("Supervised: Labels guide the algorithm",
                             "Feature 1", "Feature 2")
    elif mode == "reg":
        m = LinearRegression().fit(X_reg, y_reg)
        traces = [
            go.Scatter(x=X_reg.ravel(), y=y_reg, mode="markers",
                marker=dict(color=PALETTE["muted"], size=7, opacity=0.6), name="Data"),
            go.Scatter(x=X_reg.ravel(), y=m.predict(X_reg), mode="lines",
                line=dict(color=PALETTE["primary"], width=2.5), name="Fit"),
        ]
        layout = base_layout("Supervised: Predict a continuous target", "Feature", "Target")
    elif mode == "clu":
        from sklearn.cluster import KMeans
        km = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_moons)
        colors = [PALETTE["primary"] if li==1 else PALETTE["secondary"] for li in km.labels_]
        traces = [go.Scatter(x=X_moons[:,0], y=X_moons[:,1], mode="markers",
            marker=dict(color=colors, size=8, opacity=0.7), showlegend=False)]
        layout = base_layout("Unsupervised: Algorithm discovers clusters on its own",
                             "Feature 1", "Feature 2")
    else:
        traces = [go.Scatter(x=X_cls[:,0], y=X_cls[:,1], mode="markers",
            marker=dict(color=PALETTE["muted"], size=8, opacity=0.6), showlegend=False)]
        layout = base_layout("Unsupervised: No labels — structure must be discovered",
                             "Feature 1", "Feature 2")
    with out:
        clear_output(wait=True)
        go.Figure(data=traces, layout=layout).show()

def on_change(change): render(paradigm_dd.value)
paradigm_dd.observe(on_change, names="value")
display(VBox([paradigm_dd, out]))
render("clf")

---
## What's happening?

**Supervised learning** means every training example comes with a correct answer (a label). The algorithm learns to map inputs to those labels. Classification predicts a category; regression predicts a continuous number.

**Unsupervised learning** means no labels exist. The algorithm must find structure, groupings, or compressed representations entirely from the input data. Clustering finds natural groups; dimensionality reduction finds lower-dimensional summaries.

The key insight: these are not just different algorithms. They represent different kinds of problems, determined by what information is available to you at training time.

| Type | Has labels? | Goal | Example algorithm |
|---|---|---|---|
| Supervised — Classification | Yes | Predict a category | Logistic Regression, Decision Tree |
| Supervised — Regression | Yes | Predict a number | Linear Regression, Random Forest |
| Unsupervised — Clustering | No | Find natural groups | K-Means, DBSCAN |
| Unsupervised — Dimensionality reduction | No | Compress features | PCA, t-SNE |

---
## Real-world example: Customer segmentation

A retailer has purchase history for 10 million customers but has never labeled them as "budget shopper" or "premium buyer." Without labels, they use clustering to discover those segments from the data itself. Once segments are found and manually named, the retailer can then build a supervised classifier to assign new customers to segments instantly.

This pattern, unsupervised discovery followed by supervised deployment, is one of the most common workflows in industry.

| Type | Has labels? | Goal | Example algorithm | Real-world example |
|---|---|---|---|---|
| Supervised — Classification | Yes | Predict a category | Logistic Regression | Spam detection |
| Supervised — Regression | Yes | Predict a number | Linear Regression | House price prediction |
| Unsupervised — Clustering | No | Find natural groups | K-Means | Customer segmentation |
| Unsupervised — Dim. reduction | No | Compress features | PCA | Visualizing gene expression |

---
## Key takeaway

> **Supervised learning requires labeled examples; unsupervised learning finds structure without them — and your data determines which one you can use, not which one you prefer.**

---
*Next up: The ML Workflow — a map of every step from raw data to a deployed model*